# Importing libraries

In [1]:
# Basic libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.multioutput import MultiOutputClassifier

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss
from memory_profiler import memory_usage


# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [2]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [3]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

train = ds['train'].to_pandas()
val = ds['valid'].to_pandas()
test = ds['test'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     1556 non-null   object
 1   labels    1556 non-null   object
 2   bodyText  1556 non-null   object
dtypes: object(3)
memory usage: 36.6+ KB


# Dataset preprocessing

In [4]:
allowed_categories = ["bug", "feature", "question", "won't fix", "docs"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train['labels'] = train['labels'].apply(clean_element)
test['labels'] = test['labels'].apply(clean_element)
val['labels'] = val['labels'].apply(clean_element)

In [5]:
train = train[train['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test = test[test['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val = val[val['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train = train[train['labels'].apply(len) > 0]
test = test[test['labels'].apply(len) > 0]
val = val[val['labels'].apply(len) > 0]

In [6]:
train.rename(columns={'title': 'text'}, inplace=True)
test.rename(columns={'title': 'text'}, inplace=True)
val.rename(columns={'title': 'text'}, inplace=True)

train.drop(columns=['bodyText'], inplace=True)
test.drop(columns=['bodyText'], inplace=True)
val.drop(columns=['bodyText'], inplace=True)

train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)

train

,text,labels
0,Fix docs typo in starter files,[docs]
1,Fix typo in starter files,[docs]
2,Load models give different results from original,[question]
3,Pickle error and OOM when upgrading to 1.2.0,"[question, won't fix]"
4,val_check_interval equivalent for training los...,[won't fix]
...,...,...
410,How to implement pre-training?,[question]
411,Logging the current learning rate,[question]
412,Example of gradient accumulation documentation...,[docs]
413,Checkpooint Callback not called when training ...,[question]


In [7]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train['labels'])
val_labels_binarized = mlb.transform(val['labels'])
test_labels_binarized = mlb.transform(test['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train = pd.concat([train, train_labels_df], axis=1)
val = pd.concat([val, val_labels_df], axis=1)
test = pd.concat([test, test_labels_df], axis=1)

train = train.drop(columns=['labels'])
val = val.drop(columns=['labels'])
test = test.drop(columns=['labels'])

train

,text,bug,docs,feature,question,won't fix
0,Fix docs typo in starter files,0,1,0,0,0
1,Fix typo in starter files,0,1,0,0,0
2,Load models give different results from original,0,0,0,1,0
3,Pickle error and OOM when upgrading to 1.2.0,0,0,0,1,1
4,val_check_interval equivalent for training los...,0,0,0,0,1
...,...,...,...,...,...,...
410,How to implement pre-training?,0,0,0,1,0
411,Logging the current learning rate,0,0,0,1,0
412,Example of gradient accumulation documentation...,0,1,0,0,0
413,Checkpooint Callback not called when training ...,0,0,0,1,0


In [8]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [9]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': MultiOutputClassifier(SVC()),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultiOutputClassifier(MultinomialNB()),
        'params': {
            'estimator__alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': MultiOutputClassifier(LogisticRegression(max_iter=1000)),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': MultiOutputClassifier(GradientBoostingClassifier()),
        'params': {
            'estimator__n_estimators': [100, 150, 200],
            'estimator__criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': MultiOutputClassifier(AdaBoostClassifier()),
        'params': {
            'estimator__n_estimators': [50, 100, 150],
            'estimator__learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': MultiOutputClassifier(SGDClassifier()),
        'params': {
            'estimator__alpha': [0.0001, 0.001, 0.01],
            'estimator__penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [10]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'hamming_loss', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = list(train.columns[1:])
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [11]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train[classes])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                val_classes = val.drop(columns=['text'])

                accuracy = accuracy_score(val_classes, y_pred)
                
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val_classes.to_numpy(), y_pred, average=None, zero_division=0)

                hamm_loss = hamming_loss(val_classes.to_numpy(), y_pred)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'hamming_loss': hamm_loss,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multilabel2.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 1.3487792999949306
Peak memory usage during training: 369.73828125 MB
Prediction time: 1.9985945000080392
Peak memory usage during prediction: 369.296875 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_17488\2771957900.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 1.2528648999868892
Peak memory usage during training: 370.1875 MB
Prediction time: 1.8892155000357889
Peak memory usage during prediction: 369.734375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 1.2059348999755457
Peak memory usage during training: 370.25 MB
Prediction time: 1.888440599956084
Peak memory usage during prediction: 369.828125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 1.366499700001441
Peak memory usage during training: 372.86328125 MB
Prediction time: 1.8913718999829143
Peak memory usage during prediction: 372.36328125 MB
-------------------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 1.8209572999621741
Peak memory usage during training: 377.42578125 MB
Prediction time: 1.798165299987886
Peak memory usage during prediction: 378.40625 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 2.702402400027495
Peak memory usage during training: 69.64453125 MB
Prediction time: 1.9956818000064231
Peak memory usage during prediction: 69.6171875 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 2.19060090003768
Peak memory usage during training: 69.80859375 MB
Prediction time: 1.188628299976699
Peak memory usage during prediction: 69.45703125 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 1.6048742000130005
Peak memory usage during training: 69.703125 MB
Prediction time: 2.7195778000168502
Peak memory usage during prediction: 69.67578125 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l2'} and vectorizer CountVectorizer
Training time: 4.089170899998862
Peak memory usage during training: 69.6953125 MB
Prediction time: 4.187683500000276
Peak memory usage during prediction: 69.6953125 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l1'} and vectorizer CountVectorizer
Training time: 3.513048599998001
Peak memory usage during training: 69.703125 MB
Prediction time: 2.8447860999731347
Peak memory usage during prediction: 69.703125 MB
---------------------------------------------

# Process results

In [12]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   seed                       396 non-null    object 
 1   vectorizer                 396 non-null    object 
 2   model                      396 non-null    object 
 3   params                     396 non-null    object 
 4   accuracy                   396 non-null    float64
 5   hamming_loss               396 non-null    float64
 6   training_time              396 non-null    float64
 7   prediction_time            396 non-null    float64
 8   peak_memory_train          396 non-null    float64
 9   peak_memory_prediction     396 non-null    float64
 10  precision_class_bug        396 non-null    float64
 11  recall_class_bug           396 non-null    float64
 12  f1_class_bug               396 non-null    float64
 13  precision_class_docs       396 non-null    float64

In [13]:
results.head()

,seed,vectorizer,model,params,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,f1_class_docs,precision_class_feature,recall_class_feature,f1_class_feature,precision_class_question,recall_class_question,f1_class_question,precision_class_won't fix,recall_class_won't fix,f1_class_won't fix
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.481283,0.170053,1.348779,1.998595,369.738281,369.296875,...,0.411765,0.0,0.0,0.0,0.706250,0.933884,0.804270,0.333333,0.04878,0.085106
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.438503,0.181818,1.252865,1.889216,370.187500,369.734375,...,0.312500,0.0,0.0,0.0,0.691824,0.909091,0.785714,0.000000,0.00000,0.000000
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.438503,0.181818,1.205935,1.888441,370.250000,369.828125,...,0.312500,0.0,0.0,0.0,0.691824,0.909091,0.785714,0.000000,0.00000,0.000000
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.481283,0.171123,1.366500,1.891372,372.863281,372.363281,...,0.411765,0.0,0.0,0.0,0.710692,0.933884,0.807143,0.166667,0.02439,0.042553
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.454545,0.175401,0.718584,1.856046,373.085938,372.839844,...,0.411765,0.0,0.0,0.0,0.707006,0.917355,0.798561,0.000000,0.00000,0.000000


In [14]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,precision_class_feature,recall_class_feature,f1_class_feature,precision_class_question,recall_class_question,f1_class_question,precision_class_won't fix,recall_class_won't fix,f1_class_won't fix,f1_avg
0,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.443850,0.180749,1.101619,1.112452,273.666667,273.665365,...,0.000000,0.0,0.000000,0.674157,0.991736,0.802676,0.250000,0.024390,0.044444,0.183710
1,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.429590,0.186096,1.426724,1.608228,273.781250,273.743490,...,0.000000,0.0,0.000000,0.710918,0.914601,0.799997,0.379545,0.089431,0.144386,0.282404
2,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.351159,0.212121,1.454887,1.570159,273.790365,273.779948,...,0.158730,0.1,0.122549,0.720898,0.818182,0.766450,0.238456,0.130081,0.168212,0.318776
3,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.459893,0.175401,1.883227,1.490216,273.988281,273.979167,...,0.000000,0.0,0.000000,0.674157,0.991736,0.802676,0.400000,0.048780,0.086957,0.240426
4,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.397504,0.193226,1.697489,1.196845,274.192708,274.078125,...,0.000000,0.0,0.000000,0.715219,0.906336,0.799513,0.257576,0.081301,0.122981,0.277035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'rbf'}",3.333333,0.454545,0.173262,1.023727,1.880943,377.108073,376.580729,...,0.000000,0.0,0.000000,0.698225,0.975207,0.813793,0.000000,0.000000,0.000000,0.177044
128,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'sigm...",3.333333,0.443850,0.170053,1.454132,1.879577,376.951823,376.325521,...,0.000000,0.0,0.000000,0.725490,0.917355,0.810219,0.250000,0.024390,0.044444,0.279756
129,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'lin...",3.333333,0.358289,0.216043,1.505660,1.870638,377.289062,376.445312,...,0.200000,0.1,0.133333,0.742188,0.785124,0.763052,0.238095,0.243902,0.240964,0.368608
130,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'rbf'}",3.333333,0.427807,0.188235,1.168260,1.907671,377.278646,376.653646,...,0.000000,0.0,0.000000,0.724832,0.892562,0.800000,0.090909,0.024390,0.038462,0.244084


In [15]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train[classes])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test[classes], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test[classes], y_pred, average=None, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: DecisionTree
Best model params: {'criterion': 'entropy', 'max_features': None}
Best vectorizer: TfidfVectorizer
Best accuracy: 0.315

Class bug
Precision: 0.36
Recall: 0.20930232558139536
F1: 0.2647058823529412
Support: 43

Class docs
Precision: 0.45454545454545453
Recall: 0.38461538461538464
F1: 0.4166666666666667
Support: 26

Class feature
Precision: 0.16666666666666666
Recall: 0.25
F1: 0.2
Support: 8

Class question
Precision: 0.6884057971014492
Recall: 0.7480314960629921
F1: 0.7169811320754716
Support: 127

Class won't fix
Precision: 0.21428571428571427
Recall: 0.27906976744186046
F1: 0.24242424242424243
Support: 43



In [16]:
with open('models/best_model_sklearn_multilabel2.pkl', 'wb') as f:
    pickle.dump(pipeline, f)